# Joshi Part 7: PayOff Classes and Greeks

Based on **"The Concepts and Practice of Mathematical Finance"** by Mark S. Joshi.

This notebook covers Joshi's PayOff class hierarchy (SimpleMC3-5):
- Vanilla Call/Put payoffs
- Digital (Binary) payoffs: Cash-or-Nothing, Asset-or-Nothing
- Double Digital payoffs
- The Greeks and their financial meaning

We use both from-scratch Python implementations and the RustQuant bindings.

In [ ]:
import math
import random
from RustQuant.stochastics import GeometricBrownianMotion
from RustQuant.instruments import BlackScholesMerton, OptionType

spot = 100.0
strike = 100.0
rate = 0.05
vol = 0.20
T = 1.0
n_paths = 200_000

## 1. Payoff Functions (Joshi's PayOff Classes)

In C++, Joshi uses polymorphic classes. In Python, we use simple callables.
These mirror Joshi's `PayOffCall`, `PayOffPut`, `PayOffDigitalCall`, `PayOffDoubleDigital`.

In [ ]:
# Joshi's PayOff classes as Python callables

def payoff_call(s_t, k):
    """Vanilla call: max(S_T - K, 0)"""
    return max(s_t - k, 0.0)

def payoff_put(s_t, k):
    """Vanilla put: max(K - S_T, 0)"""
    return max(k - s_t, 0.0)

def payoff_digital_call(s_t, k):
    """Digital call (cash-or-nothing): pays 1 if S_T > K"""
    return 1.0 if s_t > k else 0.0

def payoff_digital_put(s_t, k):
    """Digital put: pays 1 if S_T < K"""
    return 1.0 if s_t < k else 0.0

def payoff_double_digital(s_t, k_low, k_high):
    """Double digital (Joshi's PayOffDoubleDigital): pays 1 if K_low < S_T < K_high"""
    return 1.0 if k_low < s_t < k_high else 0.0

def payoff_power(s_t, k, n):
    """Power option: max(S_T^n - K, 0)"""
    return max(s_t**n - k, 0.0)

## 2. Generic MC Pricer with Payoff Functions

Joshi's key insight: separate the **payoff definition** from the **pricing engine**.
We use RustQuant for fast path generation, then apply any payoff in Python.

In [ ]:
def mc_price(payoff_fn, spot, rate, vol, T, n_paths, **kwargs):
    """Generic MC pricer: generate GBM paths, apply any payoff function."""
    gbm = GeometricBrownianMotion(mu=rate, sigma=vol)
    traj = gbm.simulate(x0=spot, t_end=T, n_steps=1, n_paths=n_paths)
    
    df = math.exp(-rate * T)
    payoffs = [payoff_fn(path[-1], **kwargs) for path in traj.paths]
    return df * sum(payoffs) / len(payoffs)

# Price all payoff types
print(f"{'Payoff Type':<25} {'MC Price':>10}")
print("-" * 35)
print(f"{'Vanilla Call':<25} {mc_price(payoff_call, spot, rate, vol, T, n_paths, k=strike):>10.4f}")
print(f"{'Vanilla Put':<25} {mc_price(payoff_put, spot, rate, vol, T, n_paths, k=strike):>10.4f}")
print(f"{'Digital Call':<25} {mc_price(payoff_digital_call, spot, rate, vol, T, n_paths, k=strike):>10.4f}")
print(f"{'Digital Put':<25} {mc_price(payoff_digital_put, spot, rate, vol, T, n_paths, k=strike):>10.4f}")
print(f"{'Double Digital (90,110)':<25} {mc_price(payoff_double_digital, spot, rate, vol, T, n_paths, k_low=90.0, k_high=110.0):>10.4f}")

## 3. The Greeks

Joshi emphasizes that the Greeks are the key risk measures:

| Greek | Formula | Meaning |
|-------|---------|--------|
| Delta | $\partial C / \partial S$ | Sensitivity to spot price |
| Gamma | $\partial^2 C / \partial S^2$ | Convexity (Delta sensitivity) |
| Vega | $\partial C / \partial \sigma$ | Sensitivity to volatility |
| Theta | $\partial C / \partial T$ | Time decay |
| Rho | $\partial C / \partial r$ | Sensitivity to interest rate |

In [ ]:
# Compute Greeks analytically via RustQuant
call_bsm = BlackScholesMerton(
    underlying_price=spot, strike_price=strike,
    volatility=vol, risk_free_rate=rate, cost_of_carry=rate,
    expiry_year=2027, expiry_month=3, expiry_day=22,
    option_type=OptionType.Call,
)

print("Analytic Greeks (BSM):")
print(f"  Price  = {call_bsm.price():.6f}")
print(f"  Delta  = {call_bsm.delta():.6f}")
print(f"  Gamma  = {call_bsm.gamma():.6f}")
print(f"  Vega   = {call_bsm.vega():.6f}")
print(f"  Theta  = {call_bsm.theta():.6f}")
print(f"  Rho    = {call_bsm.rho():.6f}")

## 4. Greeks via Finite Differences (Bump-and-Revalue)

Joshi shows how to estimate Greeks by bumping parameters.
This is what practitioners use when analytic Greeks are unavailable.

In [ ]:
def make_bsm(S, K, v, r, opt_type=OptionType.Call):
    return BlackScholesMerton(
        underlying_price=S, strike_price=K, volatility=v,
        risk_free_rate=r, cost_of_carry=r,
        expiry_year=2027, expiry_month=3, expiry_day=22,
        option_type=opt_type,
    )

h = 0.01  # bump size

# Delta via central difference
delta_fd = (make_bsm(spot + h, strike, vol, rate).price() - make_bsm(spot - h, strike, vol, rate).price()) / (2 * h)

# Gamma via central difference
gamma_fd = (make_bsm(spot + h, strike, vol, rate).price() - 2 * call_bsm.price() + make_bsm(spot - h, strike, vol, rate).price()) / (h**2)

# Vega via central difference
vega_fd = (make_bsm(spot, strike, vol + h, rate).price() - make_bsm(spot, strike, vol - h, rate).price()) / (2 * h)

# Rho via central difference
rho_fd = (make_bsm(spot, strike, vol, rate + h).price() - make_bsm(spot, strike, vol, rate - h).price()) / (2 * h)

print(f"{'Greek':<8} {'Analytic':>12} {'Finite Diff':>12} {'Error':>12}")
print("-" * 44)
print(f"{'Delta':<8} {call_bsm.delta():>12.6f} {delta_fd:>12.6f} {abs(call_bsm.delta() - delta_fd):>12.8f}")
print(f"{'Gamma':<8} {call_bsm.gamma():>12.6f} {gamma_fd:>12.6f} {abs(call_bsm.gamma() - gamma_fd):>12.8f}")
print(f"{'Vega':<8} {call_bsm.vega():>12.6f} {vega_fd:>12.6f} {abs(call_bsm.vega() - vega_fd):>12.8f}")
print(f"{'Rho':<8} {call_bsm.rho():>12.6f} {rho_fd:>12.6f} {abs(call_bsm.rho() - rho_fd):>12.8f}")

## 5. Put-Call Parity

Joshi proves: $C - P = S - K e^{-rT}$. This must hold for any correct pricing model.

In [ ]:
put_bsm = make_bsm(spot, strike, vol, rate, OptionType.Put)

lhs = call_bsm.price() - put_bsm.price()
rhs = spot - strike * math.exp(-rate * T)

print("Put-Call Parity:")
print(f"  C - P            = {lhs:.6f}")
print(f"  S - K*exp(-rT)   = {rhs:.6f}")
print(f"  Difference       = {abs(lhs - rhs):.10f}")

## Summary

| Joshi Concept | Python/RustQuant Implementation |
|---------------|-------------------------------|
| `PayOffCall` / `PayOffPut` | `payoff_call()` / `payoff_put()` functions |
| `PayOffDigitalCall` | `payoff_digital_call()` |
| `PayOffDoubleDigital` | `payoff_double_digital()` |
| Polymorphic pricing | Generic `mc_price()` with callable payoffs |
| Analytic Greeks | `BlackScholesMerton.delta()`, `.gamma()`, etc. |
| Bump-and-revalue | Finite difference on BSM parameters |